In [2]:
import re
import numpy as np
import pandas as pd
import string
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import nltk

In [3]:
nltk.download('punkt')
nltk.download('stopwords')

import pandas as pd

file_path = "customer_complaints_1.csv"
df = pd.read_csv(file_path, encoding='latin1')

pd.set_option('display.max_colwidth', None)
print(df)

                          author      posted_on  rating  \
0    Alantae of Chesterfeild, MI  Nov. 22, 2016       1   
1       Vera of Philadelphia, PA  Nov. 19, 2016       1   
2    Sarah of Rancho Cordova, CA  Nov. 17, 2016       1   
3       Dennis of Manchester, NH  Nov. 16, 2016       1   
4           Ryan of Bellevue, WA  Nov. 14, 2016       1   
5            Terri of Mobile, AL   Nov. 9, 2016       1   
6   Kellie of Salt Lake City, UT   Nov. 9, 2016       1   
7      Kathleen of New Haven, CT   Nov. 6, 2016       2   
8        Shira of Bloomfield, NJ   Nov. 5, 2016       1   
9       Kristy of Alpharetta, GA   Nov. 2, 2016       1   
10           Melissa of Katy, TX   Nov. 1, 2016       1   
11        Lori of Huntsville, AL   Nov. 1, 2016       1   
12     Richard of Lauderhill, FL  Oct. 31, 2016       1   
13       Liz of Eden Prairie, MN  Oct. 31, 2016       1   
14          Meta of Thornton, CO  Oct. 30, 2016       1   
15           Emma of Medford, MA  Oct. 29, 2016       1 

[nltk_data] Downloading package punkt to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
def convert_to_lowercase(text):
    return text.lower()
df["lowercased"] = df["text"].apply(convert_to_lowercase)

pd.set_option('display.max_colwidth', None)
print(df["lowercased"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [5]:
import re

def remove_urls(text):
    return re.sub(r'https\S+|www\S+', '', text)
df["urls_removed"] = df["lowercased"].apply(remove_urls)

pd.set_option('display.max_colwidth', None)
print(df["urls_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [6]:
pip install emoji

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Removal of emojis (if any) 
import emoji 
# replace emoji with '' 
def remove_emojis(text): 
    return emoji.replace_emoji(text, replace='') 
df["emojis_removed"] = df["urls_removed"].apply(remove_emojis) 
# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["emojis_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [9]:
# Replace internet slang/chat words 
# Dictionary of slang words and their replacements 
slang_dict = { 
    "tbh": "to be honest", 
    "omg": "oh my god", 
    "lol": "laugh out loud", 
    "idk": "I don't know", 
    "brb": "be right back", 
    "btw": "by the way", 
    "imo": "in my opinion", 
    "smh": "shaking my head", 
    "fyi": "for your information", 
    "np": "no problem",
    "ikr": "I know right", 
    "asap": "as soon as possible", 
    "bff": "best friend forever", 
    "gg": "good game", 
    "hmu": "hit me up", 
    "rofl": "rolling on the floor laughing" 
}
# Function to replace slang words 
def replace_slang(text): 
    # Create a list of escaped slang words 
    escaped_slang_words = []  # Empty list to store escaped slang words
    for word in slang_dict.keys(): 
        escaped_word = re.escape(word)  # Ensure special characters are escaped 
        escaped_slang_words.append(escaped_word)  # Add to list
    # Join the words using '|' 
    slang_pattern = r'\b(' + '|'.join(escaped_slang_words) + r')\b'
    # Define a replacement function 
    def replace_match(match): 
        slang_word = match.group(0)  # Extract matched slang word 
        return slang_dict[slang_word.lower()]  # Replace with full form
    # Use regex to replace slang words with full forms 
    replaced_text = re.sub(slang_pattern, replace_match, text, flags=re.IGNORECASE)
    return replaced_text 

# Apply the function to the column 
df["slangs_replaced"] = df["emojis_removed"].apply(replace_slang)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["slangs_replaced"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [10]:
# Replace Contractions 
contractions_dict = { 
    "wasn't": "was not", 
    "isn't": "is not", 
    "aren't": "are not", 
    "weren't": "were not", 
    "doesn't": "does not", 
    "don't": "do not", 
    "didn't": "did not", 
    "can't": "cannot", 
    "couldn't": "could not", 
    "shouldn't": "should not", 
    "wouldn't": "would not", 
    "won't": "will not", 
    "haven't": "have not", 
    "hasn't": "has not", 
    "hadn't": "had not", 
    "i'm": "i am", 
    "you're": "you are", 
    "he's": "he is", 
    "she's": "she is", 
    "it's": "it is", 
    "we're": "we are", 
    "they're": "they are", 
    "i've": "i have", 
    "you've": "you have", 
    "we've": "we have", 
    "they've": "they have", 
    "i'd": "i would", 
    "you'd": "you would", 
    "he'd": "he would", 
    "she'd": "she would", 
    "we'd": "we would", 
    "they'd": "they would", 
    "i'll": "i will",
    "you'll": "you will", 
    "he'll": "he will", 
    "she'll": "she will", 
    "we'll": "we will", 
    "they'll": "they will", 
    "let's": "let us", 
    "that's": "that is", 
    "who's": "who is", 
    "what's": "what is", 
    "where's": "where is", 
    "when's": "when is", 
    "why's": "why is" 
}

# Build the regex pattern for contractions 
escaped_contractions = []  # List to store escaped contractions

for contraction in contractions_dict.keys(): 
    escaped_contraction = re.escape(contraction)  # Escape special characters (e.g., apostrophes)
    escaped_contractions.append(escaped_contraction)  # Add to list

# Join the escaped contractions with '|' 
joined_contractions = "|".join(escaped_contractions) 

# Create a regex pattern with word boundaries (\b) 
contractions_pattern = r'\b(' + joined_contractions + r')\b' 

# Compile the regex 
compiled_pattern = re.compile(contractions_pattern, flags=re.IGNORECASE)

# Define a function to replace contractions 
def replace_contractions(text): 
    # Function to handle each match found
    def replace_match(match): 
        matched_word = match.group(0)  # Extract matched contraction 
        lower_matched_word = matched_word.lower()  # Convert to lowercase
        expanded_form = contractions_dict[lower_matched_word]  #Get full form from dictionary
        return expanded_form #Return the expanded form
    
    #Apply regex substitution
    expanded_text = compiled_pattern.sub(replace_match, text)

    return expanded_text #Return modified text

# Apply the function to a DataFrame column 
df["contractions_replaced"] = df["slangs_replaced"].apply(replace_contractions)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["contractions_replaced"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [11]:
# Remove punctuations and special characters 
import string

# Function to remove punctuation 
def remove_punctuation(text): 
    return text.translate(str.maketrans('', '', string.punctuation))

# Apply the function to the column 
df["punctuations_removed"] = df["contractions_replaced"].apply(remove_punctuation)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["punctuations_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [12]:
# Remove numbers 
def remove_numbers(text): 
    return re.sub(r'\d+', '', text)  # Removes all numeric characters

# Apply the function to the column 
df["numbers_removed"] = df["punctuations_removed"].apply(remove_numbers)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["numbers_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [13]:
pip install autocorrect

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Correct spelling mistakes 
from autocorrect import Speller

# Initialize spell checker 
spell = Speller(lang='en')

# Function to correct spelling 
def correct_spelling(text): 
    return spell(text)  # Apply correction

# Apply the function to the column 
df["spelling_corrected"] = df["numbers_removed"].apply(correct_spelling)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["spelling_corrected"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [15]:
# Remove stopwords 
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already downloaded 
nltk.download('stopwords')

# Define stopwords list 
stop_words = set(stopwords.words('english'))

# Function to remove stopwords 
def remove_stopwords(text):
    words = text.split()  # Split text into words 
    filtered_words = []  # Create an empty list to store words after stopword removal

    for word in words:  # Loop through each word in the list of words
        lower_word = word.lower() #Convert the word to lowercase for uniform comparison
        if lower_word not in stop_words:  # Check if the lowercase word is NOT in the stopwords list
            filtered_words.append(word)  # If it's not a stopword, add it to the filtered list
    return " ".join(filtered_words)  # Join words back into a sentence

# Apply the function to the column 
df["stopwords_removed"] = df["spelling_corrected"].apply(remove_stopwords)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["stopwords_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
# Stemming - reduces words to their base root by chopping off suffixes 
from nltk.stem import PorterStemmer

# Initialize the stemmer 
stemmer = PorterStemmer() 

# Function to apply stemming 
def stem_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]  # Apply stemming
    return " ".join(stemmed_words)

# Apply the function 
df["stemmed_words"] = df["stopwords_removed"].apply(stem_text)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["stemmed_words"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [17]:
import nltk

# Download the required resources 
nltk.download('wordnet') # For lemmatization                     
nltk.download('omw-1.4') # WordNet lexical database                    
nltk.download('averaged_perceptron_tagger_eng')  # For POS tagging 
nltk.download('punkt_tab') # For tokenization 

# Lemmatization - reduces words to their base dictionary form (lemma)
from nltk.stem import WordNetLemmatizer 
from nltk.corpus import wordnet 
from nltk.tokenize import word_tokenize 
from nltk import pos_tag

# Initialize the lemmatizer 
lemmatizer = WordNetLemmatizer()

# Function to map NLTK POS tags to WordNet POS tags 
def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'):  # Adjective
         return wordnet.ADJ
    elif nltk_tag.startswith('V'):  # Verb 
        return wordnet.VERB
    elif nltk_tag.startswith('N'): #Noun
        return wordnet.NOUN
    elif nltk_tag.startswith('R'): #Adverb
        return wordnet.ADV
    else:
        return wordnet.NOUN #Default to noun

#Function to lemmatize text with POS tagging
def lemmatize_text(text):
    if not isinstance(text,str): #Ensure input is string
        return ""
    words = word_tokenize(text) #Tokenize text into words
    pos_tags = pos_tag(words) #get POS tags

    #Lemmatize each word with its correct POS tag
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]

    return " ".join(lemmatized_words)  # Join words back into a sentence

# Apply the function to the column 
df["lemmatized"] = df["stopwords_removed"].apply(lemmatize_text)

# Display column content without truncation 
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width 
print(df["lemmatized"])

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/fc69356b-75f4-4e92-aba8-
[nltk_data]     8e539ea45739/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [18]:
from tabulate import tabulate

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["lemmatized"])

In [20]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Display the document and its predicted cluster in a table 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(df["lemmatized"], y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

# Print top terms per cluster 
print("\nTop terms per cluster:") 
order_centroids = km.cluster_centers_.argsort()[:, ::-1] 
terms = vectorizer.get_feature_names_out() 
for i in range(k): 
    print("Cluster %d:" % i) 
    for ind in order_centroids[i, :10]: 
        print(' %s' % terms[ind]) 
    print()

Document                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [21]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

Purity: 0.5789473684210527


In [22]:
!pip install gensim

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels


In [23]:
import numpy as np 
from sklearn.cluster import KMeans 
from gensim.models import Word2Vec 
from tabulate import tabulate 
from collections import Counter

In [24]:
tokenized_dataset = [doc.split() for doc in df["lemmatized"]] 
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, 
window=5, min_count=1, workers=4)

In [25]:
X = np.array([np.mean([word2vec_model.wv[word] for word in doc.split() if word in 
word2vec_model.wv], axis=0) for doc in df["lemmatized"]])

In [26]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Tabulate the document and predicted cluster 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(df["lemmatized"], y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

Document                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [27]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

Purity: 0.8421052631578947
